# Exercise 3: Metrics and suffixes

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Evaluation and Benchmarks · version 0.1.0*

> Measure how BLEU and chrF react to single-suffix changes that alter meaning (negation, case, person) in correct English→Turkish references, then compare two decoding settings of an MT model and prepare a blinded sheet for human judges. Document metric signatures.

**Setup.** `pip install "stanza==1.11.0" sacrebleu torch transformers sentencepiece llama-cpp-python fasttext-wheel pyarrow requests "turkicnlp[hfst]==0.5.0"`; each notebook uses only part of this. Exercises 1 and 2 tag the **UD 2.17** Kazakh-KTB and Turkish-IMST test files with Stanza 1.11.0 over the gold words (models from `STANZA_RESOURCES_DIR`, files cached in `UD217_DIR` and checked against their SHA-256 hashes). Exercise 3 uses the X-WMT English–Turkish test set (TIL release, commit `7d988cc`), TurkicNLP's Apertium analyser and generator for Turkish and NLLB-200 distilled 600M (`NLLB_PATH`). Exercises 4 and 5 use Qwen2.5-1.5B-Instruct as an 8-bit GGUF file run with llama.cpp on the CPU (`QWEN_GGUF`, its tokeniser from `QWEN_TOKENIZER`) and GlotLID v3 (`GLOTLID_PATH`). Exercise 4 needs the Kazakh TUMLU-mini files (`TUMLU_DIR`, revision `3523de7`), checked against the hashes of the book's audit. Model outputs are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`).

Each edit uses a candidate from TurkicNLP's Apertium analyser and generator for Turkish. The filter retains one analysis after excluding copula readings; it does not validate the edited sentence in context. The edited sentence is scored as a hypothesis against the unedited reference. The second part translates 50 X-WMT sentences with NLLB-200 using greedy and beam search.

In [ ]:
import collections, hashlib, json, math, os, random, re, statistics, unicodedata
from pathlib import Path
import requests

CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
UD = "https://raw.githubusercontent.com/UniversalDependencies/{repo}/r2.17/{file}"
SHA256 = {   # UD 2.17 release files
    "UD_Turkish-IMST/tr_imst-ud-train.conllu": "8d8b8b581dab9362e6f2e4328aef77b9fabec1d189a199e320ba136710c7bd83",
    "UD_Turkish-IMST/tr_imst-ud-test.conllu": "fc9269bd598489b942a025b678779557266ce0c054a843e47c7db386d76d6459",
    "UD_Kazakh-KTB/kk_ktb-ud-train.conllu": "6297597914a3f319db573573f927a6577c702f5fa00dcd653162aab46026aeaa",
    "UD_Kazakh-KTB/kk_ktb-ud-test.conllu": "1100846ee50c2f970562a21dca8415a381c470ea06bdb8496ae213d59c3a3c4e",
}
UD_DIR = Path(os.environ.get("UD217_DIR", "~/.cache/ud217")).expanduser()
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    FETCHED[url] = hashlib.sha256(path.read_bytes()).hexdigest()
    return path

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

def ud_path(key):
    path = UD_DIR / key
    if not path.exists():
        repo, file = key.split("/")
        r = requests.get(UD.format(repo=repo, file=file), timeout=120); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    if sha256(path) != SHA256[key]:
        raise ValueError(f"{key} differs from the UD 2.17 release file")
    return path

Word = collections.namedtuple("Word", "id form lemma upos feats")
def read(key):
    """Sentences (lists of Word) of a UD file; multiword-token ranges and empty nodes are skipped."""
    sents, words = [], []
    for line in ud_path(key).read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words: sents.append(words)
            words = []
        elif not line.startswith("#"):
            c = line.split("\t")
            if c[0].isdigit():
                words.append(Word(int(c[0]), c[1], c[2], c[3], c[5]))
    return sents

def feats(s):
    return dict(f.split("=", 1) for f in (s or "_").split("|") if "=" in f)

def tag(lang, key):
    """Stanza 1.11 POS/feature/lemma predictions for the gold words of a UD file (cached)."""
    gold = read(key)
    path = CACHE / f"ch15-stanza-{lang}-{SHA256[key][:12]}.json"
    if not path.exists():
        import logging, stanza
        logging.getLogger("stanza").setLevel(logging.ERROR)
        nlp = stanza.Pipeline(lang, processors="tokenize,pos,lemma", tokenize_pretokenized=True, verbose=False,
                              model_dir=os.environ.get("STANZA_RESOURCES_DIR", str(Path("~/stanza_resources").expanduser())),
                              download_method=None)
        doc = nlp([[w.form for w in s] for s in gold])
        pred = [[[w.lemma or "_", w.upos or "_", w.feats or "_"] for w in s.words] for s in doc.sentences]
        path.write_text(json.dumps(pred, ensure_ascii=False))
    pred = json.loads(path.read_text())
    assert [len(s) for s in pred] == [len(s) for s in gold]
    return gold, [[Word(g.id, g.form, *p) for g, p in zip(gs, ps)] for gs, ps in zip(gold, pred)]

def wilson(k, n, z=1.959963984540054):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

# ---- Qwen2.5-1.5B-Instruct (llama.cpp, q8_0) and GlotLID, for Exercises 4 and 5 ---------------
QWEN_GGUF = os.environ.get("QWEN_GGUF", str(CACHE / "models/qwen2.5-1.5b-instruct-q8_0.gguf"))
_LLM = {}
def llm():
    if "m" not in _LLM:
        from llama_cpp import Llama
        _LLM["m"] = Llama(model_path=QWEN_GGUF, n_ctx=4096, n_threads=THREADS, seed=0, verbose=False)
    return _LLM["m"]

REPLIES = CACHE / "ch15-replies.jsonl"
_CACHED = {}
if REPLIES.exists():
    for line in REPLIES.read_text(encoding="utf-8").splitlines():
        r = json.loads(line); _CACHED[r["key"]] = r["value"]
def cached(key_obj, compute):
    key = hashlib.sha1(json.dumps(key_obj, ensure_ascii=False).encode()).hexdigest()
    if key not in _CACHED:
        _CACHED[key] = compute()
        with REPLIES.open("a", encoding="utf-8") as f:
            f.write(json.dumps({"key": key, "input": key_obj, "value": _CACHED[key]}, ensure_ascii=False) + "\n")
    return _CACHED[key]

def chat(messages, max_tokens=120):
    """Greedy reply (cached)."""
    return cached(["chat", messages, max_tokens], lambda: llm().create_chat_completion(
        messages=messages, temperature=0.0, max_tokens=max_tokens)["choices"][0]["message"]["content"].strip())

_LID = {}
def lid(text):
    if "m" not in _LID:
        import fasttext
        path = os.environ.get("GLOTLID_PATH")
        if not path:
            from huggingface_hub import hf_hub_download
            path = hf_hub_download("cis-lmu/glotlid", "model_v3.bin")
        _LID["m"] = fasttext.load_model(path)
    labels, probs = _LID["m"].predict([text.replace("\n", " ")], k=1)
    return labels[0][0].replace("__label__", ""), float(probs[0][0])

print("UD files:", ", ".join(k.split("/")[1] for k in SHA256))

In [ ]:
import hfst, sacrebleu
from sacrebleu.metrics import BLEU, CHRF
TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/xwmt/test/en-tr/"
def detok(s):
    s = re.sub(r" ([’'] ?)", lambda m: m.group(1).strip(), s)
    s = re.sub(r" ([.,!?;:%)\]»])", r"\1", s)
    return re.sub(r"([(\[«]) ", r"\1", s).strip()
en = [detok(s) for s in fetch(TIL + "en-tr.en").read_text(encoding="utf-8").splitlines()]
tr = [detok(s) for s in fetch(TIL + "en-tr.tr").read_text(encoding="utf-8").splitlines()]
import turkicnlp
from turkicnlp.resources.registry import ModelRegistry
turkicnlp.download("tur", processors=["morph"])
d = ModelRegistry.get_model_path("tur", "morph", "apertium", script="Latn")
ana, gen = hfst.HfstInputStream(str(d / "tur.automorf.hfst")).read(), hfst.HfstInputStream(str(d / "tur.autogen.hfst")).read()
TOK = re.compile(r"\w+|[^\w\s]+")
def edit(sentence, kind, rng):
    """Change one word's suffixes with the Apertium analyser and generator; None if no candidate."""
    words = TOK.findall(sentence)
    order = list(range(len(words))); rng.shuffle(order)
    for i in order:
        readings = {r for r, _ in ana.lookup(words[i]) if "+" not in r} if words[i].isalpha() and words[i].islower() else set()
        if len(readings) != 1:
            continue                                   # one analysis (ignoring copula readings) only
        r = readings.pop()
        if kind == "negation" and re.search(r"<v><(tv|iv)>", r) and "<neg>" not in r and re.search(r"<(ifi|past|prog|fut|aor)>", r):
            new = re.sub(r"(<v><(?:tv|iv)>)", r"\1<neg>", r, count=1)
        elif kind == "case" and "<n>" in r and re.search(r"<(dat|abl|loc)>", r):
            new = r.replace("<dat>", "<X>").replace("<abl>", "<dat>").replace("<loc>", "<abl>").replace("<X>", "<abl>")
        elif kind == "person" and "<v>" in r and "<p3><sg>" in r:
            new = r.replace("<p3><sg>", "<p1><sg>")
        else:
            continue
        out = gen.lookup(new)
        if out and out[0][0] != words[i]:
            return re.sub(rf"\b{re.escape(words[i])}\b", out[0][0], sentence, count=1), (words[i], out[0][0])
    return None
rng = random.Random(0)
SAMPLE = list(range(len(tr)))
pairs = collections.defaultdict(list)
for i in SAMPLE:
    for kind in ("negation", "case", "person"):
        e = edit(tr[i], kind, rng)
        if e and len(pairs[kind]) < 100:
            pairs[kind].append((i, *e))
    w = TOK.findall(tr[i])
    content = [x for x in w if x.isalpha() and len(x) > 3]
    if content and len(pairs["word dropped"]) < 100:
        x = rng.choice(content); pairs["word dropped"].append((i, re.sub(rf"\b{re.escape(x)}\b ?", "", tr[i], count=1), (x, "")))
print(f"X-WMT en-tr test: {len(tr)} sentences; each edit applied to one word of up to 100 sentences (reference = original)")
print(f"{'edit':<14}{'n':>5}{'sent chrF':>11}{'sent BLEU':>11}   example")
for kind, ps in pairs.items():
    c = statistics.mean(sacrebleu.sentence_chrf(h, [tr[i]]).score for i, h, _ in ps)
    b = statistics.mean(sacrebleu.sentence_bleu(h, [tr[i]]).score for i, h, _ in ps)
    print(f"{kind:<14}{len(ps):>5}{c:11.1f}{b:11.1f}   {ps[0][2][0]} -> {ps[0][2][1] or '(deleted)'}")

# NLLB-200 greedy vs beam search on 50 sentences, and a blinded sheet for human judges.
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
torch.set_num_threads(THREADS)
NLLB = os.environ.get("NLLB_PATH", str(Path("~/.turkicnlp/models/huggingface/facebook--nllb-200-distilled-600M").expanduser()))
def translate(sents, beams):
    def run():
        tok = AutoTokenizer.from_pretrained(NLLB, src_lang="eng_Latn"); m = AutoModelForSeq2SeqLM.from_pretrained(NLLB).eval()
        out = []
        with torch.inference_mode():
            for k in range(0, len(sents), 8):
                b = tok(sents[k:k + 8], return_tensors="pt", padding=True)
                g = m.generate(**b, forced_bos_token_id=tok.convert_tokens_to_ids("tur_Latn"), num_beams=beams, max_new_tokens=200)
                out += tok.batch_decode(g, skip_special_tokens=True)
        return out
    return cached(["nllb", sents, beams], run)
src, ref = en[:50], tr[:50]
greedy, beam = translate(src, 1), translate(src, 4)
bleu, chrf = BLEU(), CHRF()
for name, hyp in (("greedy", greedy), ("beam 4", beam)):
    print(f"NLLB {name}: BLEU {bleu.corpus_score(hyp, [ref]).score:.1f}  chrF {chrf.corpus_score(hyp, [ref]).score:.1f}")
print("signatures:", bleu.get_signature(), "|", chrf.get_signature())
diff = [i for i in range(50) if greedy[i] != beam[i]]
rng = random.Random(1); sheet = []
for i in diff:
    a, b = (greedy[i], beam[i]) if rng.random() < 0.5 else (beam[i], greedy[i])
    sheet.append({"item": i, "source": src[i], "A": a, "B": b, "key": "A=greedy" if a == greedy[i] else "A=beam"})
print(f"outputs differ for {len(diff)} of 50 sentences -> blinded A/B sheet (key kept separately); first item:")
print("  source:", sheet[0]["source"][:120]); print("  A:", sheet[0]["A"][:120]); print("  B:", sheet[0]["B"][:120])

## Worked solution

Output of `experiments/ch15_exercises/run_ch15_exercises.py`:

```
X-WMT en-tr test: 700 sentences; each edit applied to one word of up to 100 sentences (reference = original)
edit              n  sent chrF  sent BLEU   example
word dropped    100       92.3       85.3   teröristlerin -> (deleted)
negation        100       97.7       89.3   söyledi -> söylemedi
person          100       98.9       89.7   söyledi -> söyledim
case            100       96.8       85.1   olayda -> olaydan
NLLB greedy: BLEU 11.6  chrF 49.1
NLLB beam 4: BLEU 12.3  chrF 50.9
signatures: nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0 | nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|version:2.6.0
outputs differ for 46 of 50 sentences -> blinded A/B sheet (key kept separately); first item:
  source: The intention would also be to infiltrate terrorists through the waters, IB officials further note.
  A: IB yetkilileri ayrıca, teröristlerin sulardan geçmesi de amaçlanacağını belirtti.
  B: IB yetkilileri ayrıca, teröristlerin sular üzerinden sızması da amaçlanacağını belirtiyorlar.
```

**High overlap can survive a changed suffix.** Mean sentence chrF is 97.7 after adding negation, 96.8 after a case change and 98.9 after a person change. Each condition contains 100 automatically edited references. Word deletion gives 92.3 on its own selected set. These sets are not matched for sentence length or edit location, so the differences do not isolate edit severity. Word deletion can also change meaning substantially.

Apertium supplies candidate forms; it does not validate each edited sentence in context. The filter excludes analyses containing `+` before testing uniqueness, so it does not establish complete morphological disambiguation. Speakers should review the changes before they are treated as labelled meaning errors.

**Compare decoding settings, not two separately trained systems.** On the first 50 X-WMT sentences, NLLB-200 600M scores BLEU 11.6 and chrF 49.1 with greedy decoding, versus 12.3 and 50.9 with beam search. No paired uncertainty estimate or human ratings are supplied, so the higher point estimates do not establish better translation quality.

The outputs differ on 46 items. The code constructs randomised A/B rows but keeps the answer key in the same in-memory records and prints only one preview. Before annotation, export the full source and A/B texts without the key and store the key separately. A sample restricted to differing outputs estimates preferences on that subset; include identical outputs when reporting whole-sample quality.

The printed signatures describe the corpus comparison. The perturbation results are averages of sentence-level scores, whose BLEU effective-order setting differs from the corpus setting. Record those settings separately when extending the experiment.

## Your turn — extension

1. Add an edit that preserves meaning (a synonym, or a different but correct word order) and compare its metric cost with negation.
2. Rate ten sheet items yourself, blind, and compare your preferences with the metric differences.